In [1]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import loompy

In [20]:
LOOM_OUT="/rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files"

In [2]:
adata = sc.read_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/Cardio.h5ad")

In [17]:
dcm = adata[adata.obs["disease"] == "dilated cardiomyopathy"]

In [ ]:
dcm.obs["disease"]

In [ ]:
gene_names_all = dcm.var["feature_name"].values
is_symbol = ~pd.Series(gene_names_all).str.startswith("ENSG").values
gene_names_filtered = gene_names_all[is_symbol]
gene_positions = np.where(is_symbol)[0]

print(f"Cells: {dcm.shape[0]}, Genes: {dcm.shape[1]}")
print(f"Genes with proper symbols: {len(gene_names_filtered)}")
print(f"Genes without symbols (dropped): {len(gene_names_all) - len(gene_names_filtered)}")

In [ ]:
gene_series = pd.Series(gene_names_filtered)
keep_first = ~gene_series.duplicated(keep="first")

gene_names_final = gene_series[keep_first].values
final_positions = gene_positions[keep_first.values]

In [ ]:
print(f"After removing duplicates: {len(gene_names_final)}")
print(f"Example gene names: {gene_names_final[:10].tolist()}")


In [ ]:
print("Building expression matrix (genes x cells)...")

expr_cxg = dcm.X[:, final_positions]  # cells x genes

# handle either sparse or dense X
if hasattr(expr_cxg, "tocsr"):
    expr_gxc = expr_cxg.T.tocsr()
else:
    expr_gxc = expr_cxg.T

cell_ids = dcm.obs_names.values
print(f"Shape (genes x cells): {expr_gxc.shape}")
print(f"Type: {type(expr_gxc)}")

In [18]:
print(f"dtype: {expr_gxc.dtype}")

cell_info = dcm.obs[["cell_states", "donor_id", "disease", "sex", "assay"]].copy()
print(cell_info.shape)
print(cell_info.head())

dtype: float32
(142946, 5)
  cell_states donor_id                 disease     sex      assay
0        vCM2      DP2  dilated cardiomyopathy  female  10x 3' v3
1      vCM3.0      DP2  dilated cardiomyopathy  female  10x 3' v3
2      vCM1.0      DP2  dilated cardiomyopathy  female  10x 3' v3
3        vCM2      DP2  dilated cardiomyopathy  female  10x 3' v3
4      vCM1.3      DP2  dilated cardiomyopathy  female  10x 3' v3


In [16]:
dense_mtx = expr_gxc.toarray()   # do this once, keep it around

row_attrs = {"Gene": np.asarray(gene_names_final, dtype=object)}
col_attrs = {
    "CellID": np.asarray(cell_ids, dtype=object),
    "cell_states": np.asarray(cell_info["cell_states"].to_numpy(), dtype=object),
    "donor_id": np.asarray(cell_info["donor_id"].to_numpy(), dtype=object),
}

print("Writing loom...")
loompy.create(LOOM_OUT, dense_mtx, row_attrs, col_attrs)
print(f"Created: {LOOM_OUT}")

Writing loom...
Created: /rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files/expr_dcm.loom


In [24]:
norm = adata[adata.obs["disease"] == "normal"]
norm = norm[norm.obs["assay"] == "10x 3' v3"]

In [25]:
gene_names_all = norm.var["feature_name"].values
is_symbol = ~pd.Series(gene_names_all).str.startswith("ENSG").values
gene_names_filtered = gene_names_all[is_symbol]
gene_positions = np.where(is_symbol)[0]

print(f"Cells: {norm.shape[0]}, Genes: {norm.shape[1]}")
print(f"Genes with proper symbols: {len(gene_names_filtered)}")
print(f"Genes without symbols (dropped): {len(gene_names_all) - len(gene_names_filtered)}")

Cells: 99981, Genes: 32383
Genes with proper symbols: 24938
Genes without symbols (dropped): 7445


In [26]:
gene_series = pd.Series(gene_names_filtered)
keep_first = ~gene_series.duplicated(keep="first")

gene_names_final = gene_series[keep_first].values
final_positions = gene_positions[keep_first.values]

In [27]:
print("Building expression matrix (genes x cells)...")

expr_cxg = norm.X[:, final_positions]  # cells x genes

# handle either sparse or dense X
if hasattr(expr_cxg, "tocsr"):
    expr_gxc = expr_cxg.T.tocsr()
else:
    expr_gxc = expr_cxg.T

cell_ids = norm.obs_names.values
print(f"Shape (genes x cells): {expr_gxc.shape}")
print(f"Type: {type(expr_gxc)}")

Building expression matrix (genes x cells)...
Shape (genes x cells): (24922, 99981)
Type: <class 'scipy.sparse._csr.csr_matrix'>


In [28]:
print(f"dtype: {expr_gxc.dtype}")

cell_info = norm.obs[["cell_states", "donor_id", "disease", "sex", "assay"]].copy()
print(cell_info.shape)
print(cell_info.head())

dtype: float32
(99981, 5)
    cell_states donor_id disease   sex      assay
202      vCM1.0       H3  normal  male  10x 3' v3
203        vCM2       H3  normal  male  10x 3' v3
204      vCM3.0       H3  normal  male  10x 3' v3
205      vCM1.0       H3  normal  male  10x 3' v3
206      vCM1.3       H3  normal  male  10x 3' v3


In [31]:
dense_mtx = expr_gxc.toarray()   # do this once, keep it around
LOOM_OUT="/rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files/expr_norm.loom"
row_attrs = {"Gene": np.asarray(gene_names_final, dtype=object)}
col_attrs = {
    "CellID": np.asarray(cell_ids, dtype=object),
    "cell_states": np.asarray(cell_info["cell_states"].to_numpy(), dtype=object),
    "donor_id": np.asarray(cell_info["donor_id"].to_numpy(), dtype=object),
}

print("Writing loom...")
loompy.create(LOOM_OUT, dense_mtx, row_attrs, col_attrs)
print(f"Created: {LOOM_OUT}")

Writing loom...
Created: /rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files/expr_norm.loom
